# 04 — Training Walk-Forward CV

Train TCN+GRU folds with train-only scaling, HMM diagnostics, and artifact export.

In [ ]:
import torch, sys
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime → T4"
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_BASE  = '/content/drive/MyDrive/yeniBot'
DATA_DIR    = f'{DRIVE_BASE}/data'
CHECKPT_DIR = f'{DRIVE_BASE}/checkpoints'
os.makedirs(f'{DATA_DIR}/processed', exist_ok=True)
os.makedirs(CHECKPT_DIR, exist_ok=True)

In [ ]:
import os, sys
REPO_URL = 'https://github.com/umutergul74/yeniBot.git'
REPO_DIR = '/content/yenibot_repo'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
sys.path.insert(0, REPO_DIR)
print('After git pull, use Runtime → Restart session before trusting changed imports.')

In [ ]:
!pip install -q -r {REPO_DIR}/requirements.txt

In [ ]:
import yaml
with open(f'{REPO_DIR}/config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['paths']['data_dir'] = DATA_DIR
cfg['paths']['checkpoint_dir'] = CHECKPT_DIR

In [ ]:
import pandas as pd
from google.colab import runtime
from yenibot.features import filter_feature_columns, resolve_feature_profile, select_feature_columns
from yenibot.training import run_walk_forward_training

labeled = pd.read_parquet(f'{DATA_DIR}/processed/labeled_1h.parquet')
profile = resolve_feature_profile(cfg)
feature_columns = filter_feature_columns(select_feature_columns(labeled), cfg)
print('Active feature profile:', profile.get('name'))
print('Training rows:', len(labeled), 'features:', len(feature_columns))
print('Seed:', cfg['project']['random_seed'], 'deterministic:', cfg['project'].get('deterministic', False))

try:
    result = run_walk_forward_training(
        labeled,
        cfg,
        feature_columns=feature_columns,
        checkpoint_dir=CHECKPT_DIR,
    )
    print(result['report'])
finally:
    runtime.unassign()


In [ ]:
# Runtime is released by the training cell's finally block.
